In [1]:
from src import utils
from src import fix_polygons

In [ ]:
MYSQL_hostname = "127.0.0.1"
sql_username = "tist"
sql_password = input("Database Password")
sql_port = 3306
ssh_host = "sherwood.cl.cam.ac.uk"
ssh_user = "hw562"
ssh_pkey = "../keys/ssh_harveyw"
local_port = 3307

In [ ]:
datafetcher = utils.DataFetcher(MYSQL_hostname, local_port, sql_username, sql_password)

In [ ]:
location_query = """
select locationid, trackid, groupid, clusterid, loc.valid_start from tist_data.cluster cluster
join tist_data.area area using (areaid)
join tist_data.cluster2group c2g using(clusterid)
join tist_data.groups gp using (groupid)
join tist_data.group2location g2l using (groupid)
join tist_data.location loc using (locationid)
join tist_data.location_track_master ltm using (locationid)
where 
cluster.valid_end is null and
area.valid_end is null and
c2g.valid_end is null and
gp.date_deregistered is null and
g2l.valid_end is null and
loc.valid_end is null and
ltm.best = 1 and ltm.valid_end is null
and cluster.areaid = 2
;
"""


#NB removed track 492586 only has 2 points
tracks_query = """
select pointid, trackid, locationid, latitude, longitude from tist_data.cluster cluster
join tist_data.area area using (areaid)
join tist_data.cluster2group c2g using(clusterid)
join tist_data.groups gp using (groupid)
join tist_data.group2location g2l using (groupid)
join tist_data.location loc using (locationid) 
join tist_data.location_track_master ltm using (locationid)
join tist_data.location_track_sub lts using (trackid)
where 
cluster.valid_end is null and
area.valid_end is null and
c2g.valid_end is null and
gp.date_deregistered is null and
g2l.valid_end is null and
loc.valid_end is null and
ltm.best = 1 and ltm.valid_end is null
and cluster.areaid = 2
;
"""

In [ ]:
metadata_query = location_query


# Fetch dataframes from remote
metadata = utils.metadata_from_SQL(metadata_query, datafetcher)
polygons = utils.polygons_from_SQL(tracks_query, datafetcher, remove_duplicates=True)

# Ensure metadata is indexed by locationid
metadata = metadata.set_index(metadata['locationid'])

# Ensure that polygons gains a locationid column and that it is still indexed by trackid
polygons = polygons.merge(metadata, on='trackid').set_index(polygons.index)



# Pre-filter
# NB simplify treats the coordinates as if they are Cartesian but this is fine because they effectively are 
# ..at the level of a single farm. Consider the local tangent plane (sin(longitude-offset), sin(latitude-offset))
polygons['geometry'] = polygons['geometry'].apply(lambda x: x.simplify(tolerance = 0.00003, preserve_topology = False))
polygons = polygons.loc[polygons['geometry'].geom_type == 'Polygon']



# Create table of pair-wise overlaps to be resolved. The geometry column contains the intersections as Polygon or MultiPolygon 
intersections = polygons.overlay(polygons)

# Remove self-comparisons
intersections = intersections.query("trackid_1 != trackid_2")

# Remove duplicates that arise from comparing a dataframe with itself
intersections = intersections.query("trackid_1 < trackid_2")

# Resolve overlaps
fix_polygons.resolve_overlaps(intersections, metadata, polygons, dividing_points=[0, 0.3, 0.5, 0.8, 1])


# Post-filter
final = polygons.loc[(polygons['geometry'].geom_type == 'Polygon')&(polygons['removed'] == 0)].drop('trackid', axis='columns')


# Write to file
utils.gdf_to_shapefile(final, "shapefiles/test/area_2.shp")